# score_runs_qlora_test_r8 — GPT-4.1 PF/SC for QLoRA r8 on the test split
The scorer from `score_runs.ipynb`, unchanged: same break patch, same `resize_ref`, same `score_row`, same `max_workers=8`. References capped at 1024^2 and rounded to /16; generated image passed last, unresized. `with_scene=False`.

The judge is always shown `r["instruction"]` — the original test-split instruction.

Scores `gen/test/qlora_r8__step2000` into `scores/test__qlora_r8_step2000.jsonl`. Resumable; failures stay retryable, so re-run the scoring cell until it prints `to score 0`.

Section 6 gives the paired per-row delta against the prompting baseline with strata and TOST. Section 7 gives the contrast against LoRA r8 at step 1000 — the same rank in bf16, each arm at its own GPT-selected checkpoint. The delta is a selected-arm comparison, not an isolation of quantisation: the two arms differ in base precision and in training steps. NF4 on its own is isolated by the no-adapter q309-vs-bf16 probe.

488 rows = 976 calls

## 1. Repo, scorer patch

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
assert _head == COMMIT, f"repo is at {_head[:7]}, expected {COMMIT[:7]}"
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [ ]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "test"
RUN      = "qlora_r8_step2000"
BASE_RUN = "baseline"                 # prompting baseline, the floor every arm is measured against
LORA_RUN = "lora_r8_step1000"         # same rank in bf16, each arm at its own selected step

GEN    = f"{DRIVE}/gen/{DATASET}/qlora_r8__step2000"
SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl"
os.makedirs(os.path.dirname(SCORES), exist_ok=True)

have = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())
assert have, f"no generated images in {GEN}"
print(f"images : {GEN} -> {len(have)} (rows {have[0]}..{have[-1]})")
print(f"scores : {SCORES}")
man = f"{GEN}/manifest.json"
m = json.load(open(man)) if os.path.exists(man) else {}
if m:
    print(f"manifest: {m.get('gpu')} | step {m.get('step')} | rank {m.get('adapter',{}).get('rank')}")
    print(f"base    : {m.get('base')}")
else:
    print("manifest: absent")

Mounted at /content/drive
images : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/test/qlora_r8__step2000 -> 488 (rows 0..487)
scores : /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/test__qlora_r8_step2000.jsonl
manifest: NVIDIA A100-SXM4-80GB | step 2000 | rank 8
base    : {'precision': 'nf4', 'double_quant': True, 'compute_dtype': 'bfloat16', 'linear4bit_modules': 309, 'bitsandbytes': '0.50.0'}


## 3. Reference images

In [ ]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
assert max(have) < len(rows), f"image index {max(have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | scoring {len(have)}")

restored 37063 / 37063
test: 488 rows | scoring 488


## 4. Sanity — 3 rows

In [ ]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

for i in have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

row 0: PF=8  SC=8
   PF: The final image accurately depicts a young man in a sharp suit (matching the reference), sitting at a sleek conference table, intently worki
   SC: The subject in the final image (image 2) mostly matches the identity of the individual in the reference image (image 1): both have similar f

row 1: PF=4  SC=0
   PF: The second image gets several aspects of the instruction correct: the subject is a young man in a suit at a modern conference table, looking
   SC: The instruction specifies that the laptop used by the young man should be the same as the one in the first image, which features distinctive

row 2: PF=9  SC=9
   PF: The final image shows a young man in a sharp suit with straight, neatly styled hair, sitting at a modern conference table and working intent
   SC: The young man depicted in the final image has a face, hairstyle, eye shape, skin tone, and head structure that are nearly identical to those



## 5. Full pass (resumable)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])   # failures stay retryable
        except: pass

def score_row(i):
    r  = rows[i]
    gp = f"{GEN}/{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
    if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
        return {"i": i, "error": "guessed_score"}
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
            "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in have if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("scoring done ->", SCORES)
print("re-run this cell until it prints 'to score: 0'")

to score: 488 (done: 0)
  25/488
  50/488
  75/488
  100/488
  125/488
  150/488
  175/488
  200/488
  225/488
  250/488
  275/488
  300/488
  325/488
  350/488
  375/488
  400/488
  425/488
  450/488
  475/488
scoring done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/test__qlora_r8_step2000.jsonl
re-run this cell until it prints 'to score: 0'


## 6. Aggregate, paired delta vs the prompting baseline

In [ ]:
import statistics as st
from collections import defaultdict
from scipy.stats import ttest_rel, t as tdist

SIGMA_ITEM  = 0.362      # judge re-scoring SD on an identical image; RMS over sd_reps (90 rows x5, ddof=1)
SIGMA_D_REF = 0.966      # identical-config rerun, 302 screen rows
EQUIV_BOUND = 0.15       # pre-registered before any arm was generated; do not tune

def load(run):
    d, errs = {}, defaultdict(int)
    for l in open(f"{DRIVE}/scores/{DATASET}__{run}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: errs[r["error"]] += 1; continue
        d[r["i"]] = r
    return d, errs

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
cur, errs = load(RUN)
print(f"{DATASET}/{RUN}: {len(cur)} scored" + (f" | errors {dict(errs)}" if errs else ""))
print(f"  PF {st.mean([d['PF'] for d in cur.values()]):.3f}  "
      f"SC {st.mean([d['SC'] for d in cur.values()]):.3f}  "
      f"Overall {st.mean([ov(d) for d in cur.values()]):.3f}")
for key in ["n_refs", "ref_kind"]:
    g = defaultdict(list)
    for d in cur.values(): g[d[key]].append(ov(d))
    print("  by " + key + ": " + " | ".join(f"{k}: {st.mean(v):.3f} (n={len(v)})" for k, v in sorted(g.items(), key=str)))

def contrast(other, label):
    o, _ = load(other)
    I = sorted(set(cur) & set(o))
    # a row index means nothing on its own; the uid is what identifies the data row
    bad = [i for i in I if cur[i]["uid"] != o[i]["uid"]]
    assert not bad, f"uid mismatch on rows {bad[:5]} — runs are not aligned"
    print(f"\npaired delta vs {label} (n={len(I)})")
    for lab, fn in [("PF", lambda d: d["PF"]), ("SC", lambda d: d["SC"]), ("Overall", ov)]:
        d = [fn(cur[i]) - fn(o[i]) for i in I]
        tt, p = ttest_rel([fn(cur[i]) for i in I], [fn(o[i]) for i in I])
        print(f"  {lab:8s} mean {st.mean(d):+.3f}  SD {st.stdev(d):.3f}  "
              f"SE {st.stdev(d)/math.sqrt(len(d)):.3f}  t {tt:+.2f}  p {p:.3g}")

    d  = [ov(cur[i]) - ov(o[i]) for i in I]
    n  = len(d); mean = st.mean(d); sd = st.stdev(d); se = sd/math.sqrt(n); df = n-1
    print(f"\n  sigma_d (Overall)   = {sd:.3f}")
    print(f"  identical-config floor = {SIGMA_D_REF:.3f}")
    print(f"  judge-only floor       = {math.sqrt(2)*SIGMA_ITEM:.3f}   (sqrt(2) x {SIGMA_ITEM})")

    # TOST against the pre-registered bound; equivalence is claimed only on this 488-row split
    lo, hi = mean - tdist.ppf(0.95, df)*se, mean + tdist.ppf(0.95, df)*se
    p_tost = max(1 - tdist.cdf((mean + EQUIV_BOUND)/se, df), tdist.cdf((mean - EQUIV_BOUND)/se, df))
    inside = lo > -EQUIV_BOUND and hi < EQUIV_BOUND
    _, p_diff = ttest_rel([ov(cur[i]) for i in I], [ov(o[i]) for i in I])
    verdict = "EQUIVALENT" if inside else ("DIFFER" if p_diff < 0.05 else "INCONCLUSIVE")
    print(f"  90% CI [{lo:+.3f}, {hi:+.3f}] vs bound +/-{EQUIV_BOUND}  TOST p {p_tost:.3f}  -> {verdict}")
    print(f"  smallest bound provable here at 80% TOST power: {(1.645 + 1.2816)*se:.3f}")

    for key, order in [("ref_kind", ["no_up", "mixed", "all_up"]), ("n_refs", [1, 2, 3])]:
        print(f"  by {key}:")
        for lev in order:
            u = [i for i in I if cur[i][key] == lev]
            if not u: continue
            dd = [ov(cur[i]) - ov(o[i]) for i in u]
            _, pp = ttest_rel([ov(cur[i]) for i in u], [ov(o[i]) for i in u])
            print(f"    {str(lev):7s} n={len(u):3d}  Overall {st.mean(dd):+.3f} (p {pp:.3f})")

contrast(BASE_RUN, "the prompting baseline")

test/qlora_r8_step2000: 488 scored
  PF 7.990  SC 7.988  Overall 7.883
  by n_refs: 1: 7.862 (n=243) | 2: 7.946 (n=194) | 3: 7.747 (n=51)
  by ref_kind: all_up: 7.384 (n=198) | mixed: 8.153 (n=133) | no_up: 8.283 (n=157)

paired delta vs the prompting baseline (n=488)
  PF       mean -0.014  SD 0.992  SE 0.045  t -0.32  p 0.749
  SC       mean -0.100  SD 1.689  SE 0.076  t -1.31  p 0.19
  Overall  mean -0.080  SD 1.376  SE 0.062  t -1.28  p 0.2

  sigma_d (Overall)   = 1.376
  identical-config floor = 0.966
  judge-only floor       = 0.512   (sqrt(2) x 0.362)
  90% CI [-0.183, +0.023] vs bound +/-0.15  TOST p 0.130  -> INCONCLUSIVE
  smallest bound provable here at 80% TOST power: 0.182
  by ref_kind:
    no_up   n=157  Overall -0.033 (p 0.530)
    mixed   n=133  Overall -0.052 (p 0.508)
    all_up  n=198  Overall -0.136 (p 0.327)
  by n_refs:
    1       n=243  Overall -0.132 (p 0.217)
    2       n=194  Overall -0.013 (p 0.848)
    3       n= 51  Overall -0.085 (p 0.627)


## 7. Contrast against LoRA r8 at step 1000
Same rank, same data, same sampler; the arms differ in base precision, in selected training step (QLoRA 2000 from a 500-2000 grid against LoRA 1000 from a 1000-4000 grid) and in sharding, since QLoRA ran without FSDP after the mixed uint8/bf16 flat-buffer path failed. The delta is therefore a selected-arm comparison at matched rank, not a rank or capacity effect and not an isolated measure of quantisation, since the two arms sit at different checkpoints. On the val slice this ran -0.196 at matched step (p 0.042) and -0.159 best-against-best (p 0.124); the test split sizes it.

In [ ]:
contrast(LORA_RUN, "LoRA r8 @1000 (bf16, same rank)")


paired delta vs LoRA r8 @1000 (bf16, same rank) (n=488)
  PF       mean -0.074  SD 1.023  SE 0.046  t -1.59  p 0.112
  SC       mean +0.141  SD 1.667  SE 0.075  t +1.87  p 0.0616
  Overall  mean +0.040  SD 1.358  SE 0.061  t +0.65  p 0.517

  sigma_d (Overall)   = 1.358
  identical-config floor = 0.966
  judge-only floor       = 0.512   (sqrt(2) x 0.362)
  90% CI [-0.061, +0.141] vs bound +/-0.15  TOST p 0.037  -> EQUIVALENT
  smallest bound provable here at 80% TOST power: 0.180
  by ref_kind:
    no_up   n=157  Overall -0.079 (p 0.160)
    mixed   n=133  Overall -0.095 (p 0.283)
    all_up  n=198  Overall +0.224 (p 0.089)
  by n_refs:
    1       n=243  Overall +0.073 (p 0.489)
    2       n=194  Overall +0.034 (p 0.622)
    3       n= 51  Overall -0.096 (p 0.564)
